In [ ]:
import os
import time
import requests  
import pandas as pd
from bs4 import BeautifulSoup
from glob import glob

# ----------------------------------------------------
# 1. 함수들을 코드 상단에 한 번만 정의합니다.
# ----------------------------------------------------

# 데이터를 요청하는 함수 (날짜를 파라미터로 받도록 수정)
def request_estate_data(start_date, end_date, num_of_rows='15000'):
    serviceKey = '8bea1c13542aaa1626e136b275cde504cb37de3c3a31d19cd15d7759b330efdf'
    base_url = 'http://openapi.d2b.go.kr/openapi/service/BidResultInfoService'
    service_name = 'getDmstcSuccessBidResult'
    
    params = {
        'serviceKey': serviceKey,
        'opengDateBegin': start_date,
        'opengDateEnd': end_date,
        'numOfRows': num_of_rows,
        'pageNo': '1'
    }
    
    try:
        response = requests.get(base_url + service_name, params=params, timeout=30)
        response.raise_for_status()  # HTTP 오류가 발생하면 예외를 발생시킴
        soup = BeautifulSoup(response.text, 'xml')
        return soup
    except requests.exceptions.RequestException as e:
        print(f"{start_date[:4]}년 데이터 요청 중 오류 발생: {e}")
        return None

# XML 아이템 하나를 처리하는 함수
def to_extract_row(item, tag_names, col_names):
    data = {}
    for tag, col in zip(tag_names, col_names):
        tag_obj = item.find(tag)
        data[col] = tag_obj.text.strip() if tag_obj else None
    return data

# Soup 객체를 데이터프레임으로 변환하는 함수
def extract_data_and_DF(soup, tag_names, col_names):
    if not soup or not soup.find('item'):
        return pd.DataFrame() # 데이터가 없으면 빈 데이터프레임 반환
    
    item_list = [to_extract_row(item_tag, tag_names, col_names) for item_tag in soup.find_all('item')]
    return pd.DataFrame(item_list)

# ----------------------------------------------------
# 2. 반복문으로 연도별 데이터 처리 및 저장을 실행합니다.
# ----------------------------------------------------

# 공통으로 사용할 변수들
tag_names = ['bidCnt','pblancNo','pblancOdr','dcsNo','iemNo','bidNm','busiDivs','orntCode','ornt','opengDt','tbidCode','tbidName','tbidEname','tbidRptr','tbidTel','tbidAddr','tbidRate','tbidAmount']
col_names = ['참가수', '공고번호', '공고차수', '판단번호', '항목번호', '입찰명', '업무구분', '발주기관코드', '발주기관', '개찰일시', '낙찰자(업체코드)', '낙찰자(상호)', '낙찰자(영문상호)', '낙찰자(대표자)', '낙찰자(연락처)', '낙찰자(주소)', '낙찰률', '낙찰금액']
save_path = "C:/한화에어로스페이스/workspaces/Crawling project/my_projcet/입찰결과data"

# 저장 폴더가 없으면 생성
if not os.path.exists(save_path):
    os.makedirs(save_path)

# 2016년부터 2024년까지 반복
for year in range(2016, 2025):
    start_date = f'{year}0101'
    end_date = f'{year}1231'
    
    print(f"--- {year}년 데이터 처리 시작 ---")
    
    # 1. 데이터 요청
    soup = request_estate_data(start_date, end_date)
    
    # 2. 데이터프레임으로 변환
    if soup:
        df_year = extract_data_and_DF(soup, tag_names, col_names)
        
        # 3. CSV 파일로 즉시 저장
        if not df_year.empty:
            file_name = f'{year}년 물품 낙찰 결과.csv'
            full_path = os.path.join(save_path, file_name)
            df_year.to_csv(full_path, index=False, encoding='cp949')
            print(f"✅ {file_name} 저장 완료! (총 {len(df_year)}개 행)")
        else:
            print(f"⚠️ {year}년 데이터가 비어있어 파일을 저장하지 않습니다.")
    
    # API 서버에 부담을 주지 않기 위해 요청 사이에 잠시 대기
    time.sleep(1) 

print("\n🎉 모든 연도별 데이터 처리가 완료되었습니다.")

--- 2016년 데이터 처리 시작 ---
✅ 2016년 물품 낙찰 결과.csv 저장 완료! (총 10513개 행)
--- 2017년 데이터 처리 시작 ---
✅ 2017년 물품 낙찰 결과.csv 저장 완료! (총 10249개 행)
--- 2018년 데이터 처리 시작 ---
✅ 2018년 물품 낙찰 결과.csv 저장 완료! (총 10389개 행)
--- 2019년 데이터 처리 시작 ---
✅ 2019년 물품 낙찰 결과.csv 저장 완료! (총 10631개 행)
--- 2020년 데이터 처리 시작 ---
✅ 2020년 물품 낙찰 결과.csv 저장 완료! (총 10193개 행)
--- 2021년 데이터 처리 시작 ---
✅ 2021년 물품 낙찰 결과.csv 저장 완료! (총 10122개 행)
--- 2022년 데이터 처리 시작 ---
✅ 2022년 물품 낙찰 결과.csv 저장 완료! (총 10457개 행)
--- 2023년 데이터 처리 시작 ---
✅ 2023년 물품 낙찰 결과.csv 저장 완료! (총 10518개 행)
--- 2024년 데이터 처리 시작 ---
✅ 2024년 물품 낙찰 결과.csv 저장 완료! (총 10146개 행)

🎉 모든 연도별 데이터 처리가 완료되었습니다.


### 하나의 csv로 합치기

In [ ]:
path = "C:/한화에어로스페이스/workspaces/Crawling project/my_projcet/입찰결과data"
fname = '*물품 낙찰 결과.csv'
full_pattern = os.path.join(path, fname)
fileList = glob(full_pattern)
fileList

['C:/한화에어로스페이스/workspaces/Crawling project/my_projcet/입찰결과data\\2016년 물품 낙찰 결과.csv',
 'C:/한화에어로스페이스/workspaces/Crawling project/my_projcet/입찰결과data\\2017년 물품 낙찰 결과.csv',
 'C:/한화에어로스페이스/workspaces/Crawling project/my_projcet/입찰결과data\\2018년 물품 낙찰 결과.csv',
 'C:/한화에어로스페이스/workspaces/Crawling project/my_projcet/입찰결과data\\2019년 물품 낙찰 결과.csv',
 'C:/한화에어로스페이스/workspaces/Crawling project/my_projcet/입찰결과data\\2020년 물품 낙찰 결과.csv',
 'C:/한화에어로스페이스/workspaces/Crawling project/my_projcet/입찰결과data\\2021년 물품 낙찰 결과.csv',
 'C:/한화에어로스페이스/workspaces/Crawling project/my_projcet/입찰결과data\\2022년 물품 낙찰 결과.csv',
 'C:/한화에어로스페이스/workspaces/Crawling project/my_projcet/입찰결과data\\2023년 물품 낙찰 결과.csv',
 'C:/한화에어로스페이스/workspaces/Crawling project/my_projcet/입찰결과data\\2024년 물품 낙찰 결과.csv']

In [20]:
import pandas as pd
from glob import glob
import os

all_data_frames = []

for file_path in fileList:
    df = pd.read_csv(file_path, header=0, encoding='cp949')
    all_data_frames.append(df)

PBL_df = pd.concat(all_data_frames, ignore_index=True)
PBL_df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 93218 entries, 0 to 93217
Data columns (total 18 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   참가수        93218 non-null  int64  
 1   공고번호       93218 non-null  object 
 2   공고차수       93218 non-null  int64  
 3   판단번호       93218 non-null  object 
 4   항목번호       93218 non-null  object 
 5   입찰명        93218 non-null  object 
 6   업무구분       93218 non-null  object 
 7   발주기관코드     93218 non-null  object 
 8   발주기관       93218 non-null  object 
 9   개찰일시       93218 non-null  int64  
 10  낙찰자(업체코드)  93218 non-null  object 
 11  낙찰자(상호)    93218 non-null  object 
 12  낙찰자(영문상호)  67961 non-null  object 
 13  낙찰자(대표자)   93218 non-null  object 
 14  낙찰자(연락처)   93218 non-null  object 
 15  낙찰자(주소)    93218 non-null  object 
 16  낙찰률        91973 non-null  float64
 17  낙찰금액       93218 non-null  float64
dtypes: float64(2), int64(3), object(13)
memory usage: 12.8+ MB


In [24]:
PBL_df = PBL_df[PBL_df['업무구분'] != '용역']
PBL_df['개찰일시'] = pd.to_datetime(PBL_df['개찰일시'].astype(str), format='%Y%m%d%H%M')

In [25]:
PBL_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 74732 entries, 0 to 93217
Data columns (total 18 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   참가수        74732 non-null  int64         
 1   공고번호       74732 non-null  object        
 2   공고차수       74732 non-null  int64         
 3   판단번호       74732 non-null  object        
 4   항목번호       74732 non-null  object        
 5   입찰명        74732 non-null  object        
 6   업무구분       74732 non-null  object        
 7   발주기관코드     74732 non-null  object        
 8   발주기관       74732 non-null  object        
 9   개찰일시       74732 non-null  datetime64[ns]
 10  낙찰자(업체코드)  74732 non-null  object        
 11  낙찰자(상호)    74732 non-null  object        
 12  낙찰자(영문상호)  53075 non-null  object        
 13  낙찰자(대표자)   74732 non-null  object        
 14  낙찰자(연락처)   74732 non-null  object        
 15  낙찰자(주소)    74732 non-null  object        
 16  낙찰률        74599 non-null  float64       
 17

In [26]:
save_path = "C:/한화에어로스페이스/workspaces/Crawling project/my_projcet/입찰결과data"

PBL_df.to_csv(os.path.join(save_path, 'Total 물품 낙찰 결과.csv'), index=False, encoding='cp949')